In [2]:
%cd "C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph"

C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph


# Phase 1 — Data Ingestion

##  — Load TXT Transcript using LlamaIndex

In [2]:
from pathlib import Path
from llama_index.core.schema import Document

from src.config.config import load_config
from src.data_ingestion.loader import TranscriptLoader

In [15]:
config=load_config()
TRANSCRIPT_PATH=(Path(config["paths"]["raw_data"])/"Meeting_Transcript.txt")

document=TranscriptLoader.load_document(str(TRANSCRIPT_PATH))

2026-09-04 19:00:17 | INFO | src.data_ingestion.loader | Transcript loaded successfully: Meeting_Transcript.txt


## lets test metadata here 

In [14]:
from src.data_ingestion.metadata import TranscriptMetadata

In [16]:

metadata=TranscriptMetadata.extract(document=document,file_path=str(TRANSCRIPT_PATH))
metadata
    

2026-09-04 19:00:32 | INFO | src.data_ingestion.metadata | Metadata extracted for Meeting_Transcript.txt


{'file_name': 'Meeting_Transcript.txt',
 'file_type': 'txt',
 'file_size_mb': 0.0037,
 'character_count': 3917,
 'word_count': 618,
 'line_count': 93}

In [17]:
import pandas as pd 
pd.DataFrame(metadata.items(),columns=["Metric","Value"])

,Metric,Value
0,file_name,Meeting_Transcript.txt
1,file_type,txt
2,file_size_mb,0.0037
3,character_count,3917
4,word_count,618
5,line_count,93


In [18]:
from src.utils.helpers import save_json

In [19]:
INTERIM_PATH = Path(config["paths"]["interim_data"])

METADATA_PATH = INTERIM_PATH / "transcript_metadata.json"
RAW_TRANSCRIPT_PATH = INTERIM_PATH / "raw_transcript.txt"

In [20]:
save_json(
    data=metadata,
    file_path=str(METADATA_PATH),
)

print("Metadata saved successfully.")
print(METADATA_PATH)

Metadata saved successfully.
data\interim\transcript_metadata.json


In [21]:
RAW_TRANSCRIPT_PATH.write_text(
    document.text,
    encoding="utf-8",
)

print("Transcript saved successfully.")
print(RAW_TRANSCRIPT_PATH)

Transcript saved successfully.
data\interim\raw_transcript.txt


In [22]:
print(METADATA_PATH.exists())
print(RAW_TRANSCRIPT_PATH.exists())

True
True


In [23]:
from src.utils.helpers import load_json

saved_metadata = load_json(str(METADATA_PATH))

saved_metadata

{'file_name': 'Meeting_Transcript.txt',
 'file_type': 'txt',
 'file_size_mb': 0.0037,
 'character_count': 3917,
 'word_count': 618,
 'line_count': 93}

#### Created LOADER.PY  file 

#### old loader.py file 

## new loader.py file 

In [5]:
%%writefile src/data_ingestion/loader.py

from pathlib import Path
from docx import Document as DocxDocument
from llama_index.core.schema import Document


class TranscriptLoader:

    @staticmethod
    def load_document(file_path: str):
        path = Path(file_path)
        suffix = path.suffix.lower()

        if suffix == ".txt":
            text = path.read_text(encoding="utf-8")

        elif suffix == ".docx":
            doc = DocxDocument(file_path)
            text = "\n".join(p.text for p in doc.paragraphs)

        elif suffix == ".pdf":
            # Import only when a PDF is uploaded
            try:
                from PyPDF2 import PdfReader
            except ModuleNotFoundError:
                try:
                    from pypdf import PdfReader
                except ModuleNotFoundError:
                    raise ImportError(
                        "PDF support is unavailable because neither PyPDF2 nor pypdf is installed."
                    )

            reader = PdfReader(file_path)
            text = "\n".join(page.extract_text() or "" for page in reader.pages)

        else:
            raise ValueError(f"Unsupported file format: {suffix}")

        return Document(text=text)

        

Overwriting src/data_ingestion/loader.py


#### Tested Loader.pty file

In [1]:
%%writefile tests/test_loader.py

from pathlib import Path
import pytest

from llama_index.core.schema import Document

from src.data_ingestion.loader import TranscriptLoader
from src.utils.exception import ProjectException

RAW_DATA= Path("data/raw")

def test_load_text_document():
    document=TranscriptLoader.load_document(str(RAW_DATA/ "Meeting_Transcript.txt"))

    assert isinstance(document,Document)

def test_document_contains_text():
     document = TranscriptLoader.load_document(str(RAW_DATA / "Meeting_Transcript.txt"))

     assert len(document.text)>0



def test_file_not_found():
    with pytest.raises(ProjectException):
        TranscriptLoader.load_document("data/raw/not_found.txt")


def test_invalid_extension(tmp_path):
    file_path = tmp_path / "sample.csv"
    file_path.write_text("dummy data")

    with pytest.raises(ProjectException):
        TranscriptLoader.load_document(str(file_path))
    







Writing tests/test_loader.py


FileNotFoundError: [Errno 2] No such file or directory: 'tests/test_loader.py'

In [27]:
!pytest tests/test_loader.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\Users\Lenovo\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph
plugins: anyio-4.10.0, langsmith-0.10.15
collecting ... collected 4 items

tests/test_loader.py::test_load_text_document PASSED                     [ 25%]
tests/test_loader.py::test_document_contains_text PASSED                 [ 50%]
tests/test_loader.py::test_file_not_found PASSED                         [ 75%]
tests/test_loader.py::test_invalid_extension PASSED                      [100%]

============================== 4 passed in 5.18s ==============================


# Creating metadata.py

In [9]:
%%writefile src/data_ingestion/metadata.py

from pathlib import Path
import sys 

from llama_index.core.schema import Document
from src.utils.exception import ProjectException

from src.utils.logger import get_logger
logger = get_logger(__name__)

class TranscriptMetadata:

    @staticmethod
    def extract(document:Document,file_path:str)->dict:

        try:
            path=Path(file_path)

            metadata={"file_name":path.name,
                      "file_type":path.suffix.lower().replace(".",""),
                      "file_size_mb":round(path.stat().st_size/(1024*1024),4),
                      "character_count":len(document.text),
                      "word_count":len(document.text.split()),
                      "line_count":len(document.text.splitlines()),}
            
            logger.info(f"Metadata extracted for {path.name}")

            return metadata

        except Exception as error:
            logger.error(str(error))
            raise ProjectException(str(error), sys)

            


Overwriting src/data_ingestion/metadata.py


#### Testing metsdata.py

In [24]:
%%writefile tests/test_metadata.py
from pathlib import Path
from src.utils.helpers import save_json, load_json
from src.data_ingestion.loader import TranscriptLoader
from src.data_ingestion.metadata import TranscriptMetadata


RAW_DATA_PATH = Path("data/raw/Meeting_Transcript.txt")


def test_extract_metadata_returns_dictionary():
    document = TranscriptLoader.load_document(str(RAW_DATA_PATH))

    metadata = TranscriptMetadata.extract(
        document=document,
        file_path=str(RAW_DATA_PATH),
    )

    assert isinstance(metadata, dict)


def test_metadata_contains_expected_keys():
    document = TranscriptLoader.load_document(str(RAW_DATA_PATH))

    metadata = TranscriptMetadata.extract(
        document=document,
        file_path=str(RAW_DATA_PATH),
    )

    expected_keys = {
        "file_name",
        "file_type",
        "file_size_mb",
        "character_count",
        "word_count",
        "line_count",
    }

    assert expected_keys.issubset(metadata.keys())


def test_metadata_values_are_valid():
    document = TranscriptLoader.load_document(str(RAW_DATA_PATH))

    metadata = TranscriptMetadata.extract(
        document=document,
        file_path=str(RAW_DATA_PATH),
    )

    assert metadata["file_name"] == "Meeting_Transcript.txt"
    assert metadata["file_type"] == "txt"
    assert metadata["character_count"] > 0
    assert metadata["word_count"] > 0
    assert metadata["line_count"] > 0
    assert metadata["file_size_mb"] > 0

def test_save_metadata_json(tmp_path):
    document = TranscriptLoader.load_document(str(RAW_DATA_PATH))

    metadata = TranscriptMetadata.extract(
        document=document,
        file_path=str(RAW_DATA_PATH),
    )

    json_path = tmp_path / "metadata.json"

    save_json(metadata, str(json_path))

    loaded_metadata = load_json(str(json_path))

    assert metadata == loaded_metadata



Overwriting tests/test_metadata.py


In [25]:
!pytest tests/test_metadata.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\Users\Lenovo\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Lenovo\Desktop\github projects\ai-meeting-notes-analyzer-langgraph
plugins: anyio-4.10.0, langsmith-0.10.15
collecting ... collected 4 items

tests/test_metadata.py::test_extract_metadata_returns_dictionary PASSED  [ 25%]
tests/test_metadata.py::test_metadata_contains_expected_keys PASSED      [ 50%]
tests/test_metadata.py::test_metadata_values_are_valid PASSED            [ 75%]
tests/test_metadata.py::test_save_metadata_json PASSED                   [100%]

============================== 4 passed in 4.52s ==============================
